In [ ]:
import json
import time 

import pandas as pd
import numpy as np
from bs4 import BeautifulSoup

from typing import List, Optional
from pydantic import BaseModel

from supabase import create_client, Client

### Extract

In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options

options = Options()
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")

driver = webdriver.Chrome(options=options)

In [ ]:
url = "https://www.whoscored.com/matches/1983575/live/england-premier-league-2026-2027-brighton-arsenal"
driver.get(url)

In [ ]:
soup = BeautifulSoup(driver.page_source, "html.parser") 

In [ ]:
el = soup.select_one('script:-soup-contains("matchCentreData")')

In [ ]:
print(el)

In [ ]:
matchDict = json.loads(el.text.split("matchCentreData: ")[1].split(",\n")[0])
matchDict

In [ ]:
driver.quit()

In [ ]:
match_events = matchDict['events']

In [ ]:
match_events

### Transform

In [ ]:
df = pd.DataFrame(match_events)
df.head()

In [ ]:
df.count()

In [ ]:
df = df.dropna(subset='playerId')

In [ ]:
df.count()

In [ ]:
df = df.where(pd.notnull(df), None)

In [ ]:
df.columns

In [ ]:
df = df.rename(
    {
        "eventId": "event_id",
        "teamId": "team_id",
        "expandedMinute": "expanded_minute",
        "outcomeType": "outcome_type",
        "satisfiedEventsTypes": "satisfied_events_types",
        "isTouch": "is_touch",
        "playerId": "player_id",
        "endX": "end_x",
        "endY": "end_y",
        "goalMouthZ": "goal_mouth_z",
        "goalMouthY": "goal_mouth_y",
        "isShot": "is_shot",
        "bolckedX": "blocked_x",
        "bolckedY": "blocked_y",
        "relatedEventId": "related_event_id",
        "relatedPlayerId": "related_player_id",
        "isGoal": "is_goal",
        "cardType": "card_type"
    },
    axis = 1
)
df.columns

In [ ]:
df['period_display_name'] = df['period'].apply(lambda x: x['displayName'])
df['outcome_type_display_name'] = df['outcome_type'].apply(lambda x: x['displayName'])
df['type_display_name'] = df['type'].apply(lambda x: x['displayName'])

In [ ]:
df.loc[:, ["period_display_name", "outcome_type_display_name", "type_display_name"]]

In [ ]:
df = df.drop(columns=["period","type", "outcome_type"])

In [ ]:
df.columns

In [ ]:
df = df.rename({"blockedX": "blocked_x", "blockedY": "blocked_y"}, axis = 1)

In [ ]:
df = df[['id', 'event_id', 'minute', 'second', 'team_id', 'x', 'y',
       'expanded_minute', 'qualifiers', 'is_touch',
       'player_id', 'end_x', 'end_y', 'goal_mouth_z', 'goal_mouth_y',
       'is_shot', 'blocked_x', 'blocked_y', 'is_goal', 'card_type', 'period_display_name',
       'outcome_type_display_name', 'type_display_name']]

In [ ]:
df.dtypes

In [ ]:
df[['id', 'event_id','minute', 'team_id', 'player_id']] = df[['id', 'event_id','minute', 'team_id', 'player_id']].astype(int)
df[['second','x','y','end_x', 'end_y']] = df[['second','x','y','end_x', 'end_y']].astype(float)
df[["is_shot", "is_goal", "card_type"]] = df[["is_shot", "is_goal", "card_type"]].astype(bool)

In [ ]:
df['is_goal'] = df['is_goal'].fillna(False)
df['is_shot'] = df['is_shot'].fillna(False)

In [ ]:
for col in df.columns:
    if df[col].dtype == np.float64 or df[col].dtype == np.float32:
        df[col] = np.where(
            np.isnan(df[col]),
            None,
            df[col]
        )

In [ ]:
df.iloc[0].to_dict()

In [ ]:
class MatchEvent(BaseModel):
    id: int
    event_id: int
    minute: int
    second: Optional[float] = None
    team_id: int
    x: float
    y: float
    expanded_minute: int
    qualifiers: list[dict]

    is_touch: bool
    player_id: int
    end_x: Optional[float] = None
    end_y: Optional[float] = None
    goal_mouth_z: Optional[float] = None
    goal_mouth_y: Optional[float] = None
    is_shot: bool
    blocked_x: Optional[float] = None
    blocked_y: Optional[float] = None
    is_goal: bool
    card_type: bool

    period_display_name: str
    outcome_type_display_name: str
    type_display_name: str

In [ ]:
for x in df.to_dict(orient="records"):
    try:
        MatchEvent(**x).model_dump()
    except Exception as e:
        print(e)
        break

In [ ]:
df.iloc[59].to_dict()

### Load

In [ ]:
supabase_password = ""

In [ ]:
project_url = ""
api_key = ""

In [ ]:
def insert_match_events(df, supabase):
    events = [MatchEvent(**x).model_dump() for x in df.to_dict(orient="records")]
    execution = supabase.table('match_events').upsert(events).execute()

In [ ]:
supabase = create_client(project_url, api_key)

In [ ]:
insert_match_events(df, supabase)

In [ ]:
team_info = []
team_info.append({
    'team_id': matchDict['home']['teamId'],
    'name': matchDict['home']['name'],
    'country_name': matchDict['home']['countryName'],
    'manager_name': matchDict['home']['managerName'],
    'players': matchDict['home']['players'],
})

team_info.append({
    'team_id': matchDict['away']['teamId'],
    'name': matchDict['away']['name'],
    'country_name': matchDict['away']['countryName'],
    'manager_name': matchDict['away']['managerName'],
    'players': matchDict['away']['players'],
})

In [ ]:
matchDict['home']['players']

In [ ]:
class Player(BaseModel):
    player_id: int
    shirt_no: int
    name: str
    age: int
    position: str
    team_id: int

In [ ]:
def insert_players(team_info, supabase):
    players = []
    for team in team_info:
        for player in team['players']:
            players.append({
                'player_id': player['playerId'],
                'team_id': team['team_id'],
                'shirt_no': player['shirtNo'],
                'name': player['name'],
                'position': player['position'],
                'age': player['age']
            })
    execution = supabase.table("players").upsert(players).execute()

In [ ]:
insert_players(team_info, supabase)

In [ ]:
import psycopg2

conn = psycopg2.connect(
    user = "",
    password = "",
    host = "",
    port = ,
    database = ""
)

In [ ]:
cursor = conn.cursor()

In [ ]:
cursor.execute("""
   SELECT * FROM players 
""")

In [ ]:
records = cursor.fetchall()

In [ ]:
records

In [ ]:
df = pd.DataFrame(records, columns=[desc[0] for desc in cursor.description])
df.head()

### Putting It All Togther to get the Data for all the mathches in the 2026 season

In [ ]:
class MatchEvent(BaseModel):
    id: int
    event_id: int
    minute: int
    second: Optional[float] = None
    team_id: int
    player_id: int
    x: float
    y: float
    end_x: Optional[float] = None
    end_y: Optional[float] = None
    qualifiers: List[dict]
    is_touch: bool
    blocked_x: Optional[float] = None
    blocked_y: Optional[float] = None
    goal_mouth_z: Optional[float] = None
    goal_mouth_y: Optional[float] = None
    is_shot: bool
    card_type: bool
    is_goal: bool
    type_display_name: str
    outcome_type_display_name: str
    period_display_name: str

def insert_match_events(df, supabase):
    events = [
        MatchEvent(**x).dict()
        for x in df.to_dict(orient='records')
    ]
    
    execution = supabase.table('match_events').upsert(events).execute()

class Player(BaseModel):
    player_id: int
    shirt_no: int
    name: str
    age: int
    position: str
    team_id: int

def insert_players(team_info, supabase):
    players = []
    for team in team_info:
        for player in team['players']:
            players.append({
                'player_id': player['playerId'],
                'team_id': team['team_id'],
                'shirt_no': player['shirtNo'],
                'name': player['name'],
                'position': player['position'],
                'age': player['age']
            })
            
    execution = supabase.table('players').upsert(players).execute()

supabase_password = ''
project_url = ''
api_key = ''

supabase = create_client(project_url, api_key)

driver = webdriver.Chrome()

def scrape_match_events(whoscored_url, driver):
    
    driver.get(whoscored_url)
    
    soup = BeautifulSoup(driver.page_source, 'html.parser')
    
    element = soup.select_one('script:-soup-contains("matchCentreData")')
    
    matchdict = json.loads(element.text.split("matchCentreData: ")[1].split(',\n')[0])
    
    match_events = matchdict['events']
    
    df = pd.DataFrame(match_events)
    
    df.dropna(subset='playerId', inplace=True)
    
    df = df.where(pd.notnull(df), None)
    
    df = df.rename(
    {
        'eventId': 'event_id',
        'expandedMinute': 'expanded_minute',
        'outcomeType': 'outcome_type',
        'isTouch': 'is_touch',
        'playerId': 'player_id',
        'teamId': 'team_id',
        'endX': 'end_x',
        'endY': 'end_y',
        'blockedX': 'blocked_x',
        'blockedY': 'blocked_y',
        'goalMouthZ': 'goal_mouth_z',
        'goalMouthY': 'goal_mouth_y',
        'isShot': 'is_shot',
        'cardType': 'card_type',
        'isGoal': 'is_goal'
    },
        axis=1
    )
    
    df['period_display_name'] = df['period'].apply(lambda x: x['displayName'])
    df['type_display_name'] = df['type'].apply(lambda x: x['displayName'])
    df['outcome_type_display_name'] = df['outcome_type'].apply(lambda x: x['displayName'])
    
    df.drop(columns=["period", "type", "outcome_type"], inplace=True)
    
    if 'is_goal' not in df.columns:
        df['is_goal'] = False
        
    if 'is_card' not in df.columns:
        df['is_card'] = False
        df['card_type'] = False
        
    df = df[~(df['type_display_name'] == "OffsideGiven")]
    
    df = df[[
        'id', 'event_id', 'minute', 'second', 'team_id', 'player_id', 'x', 'y', 'end_x', 'end_y',
        'qualifiers', 'is_touch', 'blocked_x', 'blocked_y', 'goal_mouth_z', 'goal_mouth_y', 'is_shot',
        'card_type', 'is_goal', 'type_display_name', 'outcome_type_display_name',
        'period_display_name'
    ]]
    
    df[['id', 'event_id', 'minute', 'team_id', 'player_id']] = df[['id', 'event_id', 'minute', 'team_id', 'player_id']].astype(np.int64)
    df[['second', 'x', 'y', 'end_x', 'end_y']] = df[['second', 'x', 'y', 'end_x', 'end_y']].astype(float)
    df[['is_shot', 'is_goal', 'card_type']] = df[['is_shot', 'is_goal', 'card_type']].astype(bool)
    
    df['is_goal'] = df['is_goal'].fillna(False)
    df['is_shot'] = df['is_shot'].fillna(False)
    
    for column in df.columns:
        if df[column].dtype == np.float64 or df[column].dtype == np.float32:
            df[column] = np.where(
                np.isnan(df[column]),
                None,
                df[column]
            )
            
            
    insert_match_events(df, supabase)
    
    
    team_info = []
    team_info.append({
        'team_id': matchdict['home']['teamId'],
        'name': matchdict['home']['name'],
        'country_name': matchdict['home']['countryName'],
        'manager_name': matchdict['home']['managerName'],
        'players': matchdict['home']['players'],
    })

    team_info.append({
        'team_id': matchdict['away']['teamId'],
        'name': matchdict['away']['name'],
        'country_name': matchdict['away']['countryName'],
        'manager_name': matchdict['away']['managerName'],
        'players': matchdict['away']['players'],
    })
    
    insert_players(team_info, supabase)
    
    return print('Success')

    

driver.get('https://www.whoscored.com/teams/211/fixtures/england-brighton')
time.sleep(3)

soup = BeautifulSoup(driver.page_source, 'html.parser')

all_urls = soup.select('a[href*="\/live\/"]')

all_urls = list(set([
    'https://www.whoscored.com' + x.attrs['href']
    for x in all_urls
]))

all_urls

for url in all_urls:
    print(url)
    scrape_match_events(
        whoscored_url=url,
        driver=driver
    )
    
    time.sleep(2)
    